# ex_001 - exploratory data analysis

Separability of synthetic correct vs compensated form and class-conditional
feature distributions. Run `python run_ex001.py` from the `ml/` directory first
to generate the data and train the model.

In [ ]:
import pathlib, sys
p = pathlib.Path.cwd()
while not (p / 'features').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from features.extract import DATA_DIR, FEATURE_COLUMNS

sess = pd.read_parquet(DATA_DIR / 'ex_001_session_features.parquet')
print(sess.shape)
sess['label'].value_counts()

## Single-feature separability
No single feature should already solve the task (AUC ~ 1.0); if one does, the
synthetic labels are a disguised threshold and the model would just relearn it.

In [ ]:
from training.train_ex001 import univariate_separability
X = sess[FEATURE_COLUMNS]
y = sess['label'].to_numpy(dtype=int)
sep = univariate_separability(X, y)
sep

## Class-conditional distributions (top features)

In [ ]:
top = sep['feature'].head(6).tolist()
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, f in zip(axes.ravel(), top):
    good = sess[sess.label == 0][f].dropna()
    comp = sess[sess.label == 1][f].dropna()
    ax.boxplot([good, comp], labels=['good', 'compensated'])
    ax.set_title(f)
fig.tight_layout()

Model performance (leave-one-subject-out) vs the majority-class and rule-based
baselines, the calibration curve, and feature importances are written to
`training/out/` by `train_ex001.py`. Results are a feasibility demonstration on
synthetic data, not clinical validation.